# 구조화 생성(Structured Generation) 도입 검토 보고서
## `outlines` / vLLM structured outputs — 몽글마을 LLM 제어 개선

**작성 맥락:** 현재 SFT 플래너 모델의 출력 제어(특히 **중국어(한자) 누출**, **JSON 포맷 실패**)를 프롬프트 애원 + 사후 필터 + 재시도로 *우회* 중. 이를 디코딩 단계 제약(constrained decoding)으로 근본 해결할 수 있는지 검토한다.

> 본 노트북은 **발표자료 작성용 근거 문서**다. 로컬(GPU 없음)에서 돌아가는 셀은 실제 실행 가능하며, 모델 추론 셀은 RunPod/GPU에서 실행한다.

---
### TL;DR
1. `outlines`는 **로짓 제어가 가능한 로컬 서빙(transformers/vLLM/llama.cpp)** 에서만 동작. Claude/OpenAI 같은 외부 API엔 적용 불가.
2. 우리는 이미 **vLLM**으로 자체 서빙(`runpod_workers/llm/pipeline.py`) 중이고, **vLLM의 structured outputs 백엔드가 곧 outlines/xgrammar**다 → 대개 새 의존성 없이 `StructuredOutputsParams` 한 줄로 outlines를 쓰는 셈.
3. 최우선 적용처: **(후보1) 중국어 차단** + **(후보2) JSON 강제**. 그 외 날짜형식·enum·슬롯스키마 등 후보 4종.
4. PoC: `baseline` vs `vllm_native` vs `outlines_direct` 3변종을 동일 프롬프트셋에 돌려 `json_ok / cjk / strict / latency` 비교.

## 1. 배경 — 지금은 무엇을 '우회'하고 있나

자체 서빙 모델(Qwen2.5-7B + LoRA SFT 플래너)의 출력은 **자유 생성**이라, 원하는 제약을 모델 *바깥*에서 사후 처리로 메우고 있다.

| 우회 영역 | 현재 처리 방식 | 코드 위치 |
|---|---|---|
| 중국어(한자) 누출 | 한글 음절 비율 ≥0.5 검사 후 **거부**, `[가-힣]` 정규식 검사, "반드시 한국어로만" 프롬프트 애원 | `agents/todo_creation/planner/nodes/validate.py` (C2), `agents/feed_generation/nodes/validate_caption.py:9`, `.../assemble_caption_ctx.py:19` |
| JSON 포맷 실패 | 코드펜스 제거 정규식 + `json.loads` try/except + **재주입 재시도 2회** | `adapters/todo_creation/qwen_llm.py:53-64, 81, 151` |
| 연도 깨짐(year corruption) | API 후처리 보정 | (메모리: v2b 어댑터 잔여 작업) |
| intent/plan_kind 비정상값 | 파싱 후 fallback 강제, `isinstance` 가드 | `qwen_llm.py:91-95`, `judge_sufficiency` 가드 |

**문제의 본질:** 모델이 *틀린 토큰을 뽑은 뒤* 우리가 잡는 구조 → 실패율(메모리상 parse ~80%)·재시도 비용·코드 복잡도가 누적된다.

## 2. outlines / 제약 디코딩이란

**제약 디코딩(constrained decoding):** 매 토큰 생성 시 문법/스키마/정규식에 **맞지 않는 토큰의 로짓을 -inf로 마스킹**해, 애초에 규칙 위반 출력이 *생성 불가*하게 만든다.

- JSON Schema → 항상 유효한 JSON
- 정규식 → 정해진 패턴만 (예: `\d{4}-\d{2}-\d{2}`, 또는 *한자 제외* character class)
- enum/choice → 정해진 후보 중 하나

### ★ 핵심 전제 (추천 방향을 바꾸는 지점)
우리 서빙은 vLLM이고, **vLLM의 structured outputs는 내부적으로 outlines(또는 xgrammar)를 엔진으로 사용**한다.

- **경로 A — vLLM 내장(권장):** `SamplingParams(structured_outputs=...)` → 새 의존성 0, outlines 간접 사용.
- **경로 B — outlines 직접:** `outlines.from_vllm_offline(llm)` → vLLM 래퍼가 안 주는 세밀 제어(필드단위 정규식 합성 등)가 필요할 때.

## 3. 적용 후보 (레버리지 순)

| # | 후보 | 효과 | 대체되는 우회 코드 |
|---|---|---|---|
| **1** | **중국어(한자) 차단** | ★★★ 네 핵심 통증 | 한글비율 C2 거부, `_KOREAN_RE`, "한국어로만" 프롬프트 |
| **2** | **JSON 강제** | ★★★ parse 80%→~100% | 코드펜스 제거, `_*_REINFORCE`, `_complete_json_with_retry` |
| **3** | 날짜 `YYYY-MM-DD` 형식 | ★★ year corruption 원천 차단 | API 후처리 보정 |
| **4** | intent/plan_kind enum·choice | ★★ 쉬움/확실 | fallback 강제, isinstance 가드 |
| **5** | 슬롯 스키마 뱅크 기반 생성 | ★ 구조 적합 | (신규, D2 설계와 정합) |
| **6** | feed 캡션 길이/형식 | ☆ 과설계 주의 | `validate_caption` 일부 |

### 적용 제외 (중요)
- **Claude/OpenAI API 호출**: outlines 불가. 이미 `with_structured_output(json_schema, strict=True)`로 API-side 제약 사용 중 → 그대로 둔다 (`adapters/*/openai_llm.py`).
- 캐릭터 **이미지 생성**: 해당 없음.

> 본 PoC 범위: **후보 1 + 2** (중국어 차단 + JSON 강제). 후보 3은 스키마에 미리 포함(날짜 pattern).

## 4. 두 경로 비교 — 현재 API (문서 확인됨, 2026-06)

### 경로 A: vLLM 내장 structured outputs (offline `LLM.generate`)
```python
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams

so = StructuredOutputsParams(json=ConstrainedSplit.model_json_schema())   # 또는 regex=/choice=/grammar=
params = SamplingParams(temperature=0.1, max_tokens=800, structured_outputs=so)
out = llm.generate([prompt], params, lora_request=lora)
```
*(구버전 `GuidedDecodingParams`에서 `StructuredOutputsParams`로 네이밍 변경됨.)*

### 경로 B: outlines 직접 (offline vLLM 래핑)
```python
import outlines
model = outlines.from_vllm_offline(LLM(...))
result = model(prompt, output_type=ConstrainedSplit, sampling_params=params)  # Pydantic / JsonSchema / Regex
```

**비교에서 실제로 갈리는 지점:** vLLM 기본 백엔드(xgrammar)는 JSON string `pattern` 지원이 부분적 → **후보1(한자 금지 pattern)이 강제 안 될 수 있음**. 그러면 outlines 백엔드 강제 또는 경로 B가 정당화된다. PoC의 `cjk%` 컬럼이 이를 판별한다.

## 5. PoC 설계

`scripts/outlines_poc/` (production task-splitter 경로를 그대로 재현):

```
scripts/outlines_poc/
├── constrained.py   # SplitResult 미러 제약 스키마 + CJK 탐지기
├── eval_prompts.py  # 한국어 평가셋 12개 (한자 누출 잦은 케이스 포함)
└── run.py           # 3변종 실행 → 지표 표 출력
```

**3 변종**
| 변종 | 설명 | 의존성 |
|---|---|---|
| `baseline` | 현재: 자유생성 → 펜스제거 → json.loads → 검증/재시도 | - |
| `vllm_native` | `StructuredOutputsParams(json=schema)` | 0 (vLLM 내장) |
| `outlines_direct` | `from_vllm_offline` + `output_type=ConstrainedSplit` | `outlines` |

**지표**
- `json_ok%` : production이 수용할 JSON인가 (CJK 무관)
- `cjk%` : 한자/가나 누출률 (낮을수록 좋음 = 후보1 효과)
- `strict%` : 제약 스키마(CJK 금지 포함) 완전 통과
- `mean_ms` : 평균 생성 지연

## 6. 제약 스키마 — 핵심 코드 (로컬 실행 가능, GPU 불필요)

후보1(한자 차단)과 후보2(JSON)를 **단일 Pydantic 소스**로 표현한다. `pattern`이 `model_json_schema()`에 실려 vLLM/outlines 양쪽이 같은 스키마를 쓴다.

In [ ]:
import sys; sys.path.insert(0, ".")
from scripts.outlines_poc.constrained import (
    ConstrainedSplit, split_json_schema, contains_cjk,
    TITLE_PATTERN, DATE_PATTERN,
)
print("title 허용 pattern:", TITLE_PATTERN)
print("date  pattern     :", DATE_PATTERN)
import json
print("\nJSON Schema (vLLM/outlines 공용):")
print(json.dumps(split_json_schema(), ensure_ascii=False, indent=2)[:900])

### 6.1 후보1 검증 — 한자/가나는 탐지, 한글은 통과

In [ ]:
for s in ["발표 준비", "계획准备", "会议 준비", "レポート 작성", "test 123 ()"]:
    print(f"{'CJK!' if contains_cjk(s) else 'ok  '}  {s}")

### 6.2 제약 스키마가 한자 title을 거부하는지 (Pydantic 레벨)

In [ ]:
import json
good = '{"intent":"plan","tasks":[{"title":"발표 준비","due_date":"2026-06-20","tags":["학습"]}]}'
cjk  = '{"intent":"plan","tasks":[{"title":"会议 준비","due_date":"2026-06-20","tags":["업무"]}]}'

print("정상:", ConstrainedSplit.model_validate_json(good))
try:
    ConstrainedSplit.model_validate_json(cjk)
    print("한자 통과됨 (문제!)")
except Exception as e:
    print("한자 거부 OK ->", type(e).__name__)

## 7. 측정 실행 (RunPod / GPU)

> ⚠️ 아래 셀은 GPU + Qwen2.5-7B + LoRA가 필요하다. 로컬(macOS)에서는 실행되지 않는다.

```bash
cd <repo-root>
export LORA_REPO_ID=bigmooon/qwen2.5-7b-mongle-planner-ko-lora HF_TOKEN=...
python -m scripts.outlines_poc.run                       # 세 변종 전부
python -m scripts.outlines_poc.run --variants vllm_native outlines_direct
```

In [ ]:
# (RunPod에서 실행) 노트북 안에서 직접 돌리려면:
# import runpy, sys
# sys.argv = ["run"]
# runpy.run_module("scripts.outlines_poc.run", run_name="__main__")
print("GPU 환경에서 위 주석을 해제해 실행")

## 8. 결과 (RunPod 실행 후 채움)

실행 출력(`variant / json_ok% / cjk% / strict% / mean_ms`)을 아래 표에 옮긴다.

| variant | json_ok% | cjk% | strict% | mean_ms |
|---|---|---|---|---|
| baseline | _ | _ | _ | _ |
| vllm_native | _ | _ | _ | _ |
| outlines_direct | _ | _ | _ | _ |

### 예상(가설 — 미측정, 발표 시 실측치로 교체할 것)
- `baseline`: json_ok ~80%, cjk **>0%** (한자 누출 존재), strict 낮음
- `vllm_native` / `outlines_direct`: json_ok **~100%**, strict **~100%**
- `cjk%`: 백엔드가 string `pattern`을 강제하면 **0%**. `vllm_native`가 0이 안 되면 → outlines 백엔드/경로B 필요 신호.
- latency: 제약 디코딩은 마스킹 오버헤드로 baseline 대비 소폭 증가 가능(측정 필요).

## 9. 권장 로드맵
1. **PoC 측정** (본 노트북) → `cjk%`/`json_ok%`로 경로 A 충분성 판정.
2. **production 적용 (후보1+2):** `runpod_workers/llm/pipeline.py`의 `generate()`에 `structured_outputs` 주입. plan_kind/task-split 경로부터, 새 의존성 0(경로 A).
3. 효과 확인 후 **후보3·4**(날짜·enum) 확장 → 어댑터의 재시도·가드·후처리 코드 제거.
4. 경로 A로 부족한 세밀 제어가 필요할 때만 **outlines 직접 도입**.

## 10. 리스크 / 미해결
- vLLM 빌드의 structured-outputs **백엔드 선택 API**(xgrammar↔outlines 강제) — 우리 버전 확인 필요.
- `from_vllm_offline`의 **LoRA(`lora_request`) 전달** 가능 여부 — `run.py`에서 graceful degrade 처리.
- xgrammar의 **JSON string `pattern` 부분 지원** — 후보1 강제 여부는 PoC `cjk%`로 검증.
- 제약 디코딩 **latency 오버헤드** — 서버리스 콜드스타트 환경에서 측정 필요.

---
*근거 파일: `scripts/outlines_poc/{constrained,eval_prompts,run}.py`, `runpod_workers/llm/pipeline.py`, `adapters/todo_creation/qwen_llm.py`*